# 05 - Dashboard Prep: Export Tidy CSVs for Power BI

**Inputs:** `../data/cleaned_readable.csv`, `../data/IDS_mapping.csv`, `../data/model_metrics.csv`, `../data/top_features.csv`  
**Output folder:** `../data/powerbi/`

"Tidy" means: **one row per observation, one column per variable, one value per cell.** Power BI works best with tables like this.

| File | One row = | Used on page |
|------|-----------|--------------|
| `patients.csv` | one patient | 1, 2 |
| `readmission_by_group.csv` | one group of one variable (long format) | 2 |
| `model_metrics.csv` | one model x one metric (long format) | 3 |
| `top_features.csv` | one model x one feature | 3 |

The plan for the 3 dashboard pages is in `../docs/dashboard_plan.md`.

## Step 1: Load

In [1]:
import pandas as pd
import os

df = pd.read_csv("../data/cleaned_readable.csv")

# create the output folder if it does not exist yet
os.makedirs("../data/powerbi", exist_ok=True)

print("Patients:", len(df))

Patients: 69970


## Step 2: Replace ID Codes with Their Descriptions

Power BI users should see "Emergency", not "1". The three lookup tables come from `IDS_mapping.csv` (same split as in `sql/load_data.py`).

In [2]:
admission_type = pd.read_csv("../data/IDS_mapping.csv", nrows=8)
discharge = pd.read_csv("../data/IDS_mapping.csv", skiprows=10, nrows=30)
admission_source = pd.read_csv("../data/IDS_mapping.csv", skiprows=42)

# rename the description column so each one has its own name after merging
admission_type = admission_type.rename(columns={"description": "admission_type"})
discharge = discharge.rename(columns={"description": "discharge_disposition"})
admission_source = admission_source.rename(columns={"description": "admission_source"})

# remove extra spaces at the start of some descriptions
admission_source["admission_source"] = admission_source["admission_source"].str.strip()

# left merge = keep every patient, add the matching description
df = df.merge(admission_type, on="admission_type_id", how="left")
df = df.merge(discharge, on="discharge_disposition_id", how="left")
df = df.merge(admission_source, on="admission_source_id", how="left")

# the file writes NULL for some codes, which pandas reads as missing
for col in ["admission_type", "discharge_disposition", "admission_source"]:
    df[col] = df[col].fillna("Unknown")

df[["admission_type_id", "admission_type", "discharge_disposition_id", "discharge_disposition"]].head()

,admission_type_id,admission_type,discharge_disposition_id,discharge_disposition
0,2,Urgent,1,Discharged to home
1,3,Elective,3,Discharged/transferred to SNF
2,1,Emergency,1,Discharged to home
3,1,Emergency,1,Discharged to home
4,2,Urgent,1,Discharged to home


## Step 3: Add Band Columns for Charts

Small bands are easier to chart than raw numbers.

In [3]:
# prior inpatient visits: 0, 1, 2, 3+
df["inpatient_band"] = df["number_inpatient"].clip(upper=3).astype(int).astype(str)
df.loc[df["inpatient_band"] == "3", "inpatient_band"] = "3+"

# length of stay band
df["stay_band"] = pd.cut(df["time_in_hospital"], bins=[0, 3, 6, 9, 12],
                         labels=["1-3 days", "4-6 days", "7-9 days", "10-12 days"])

df[["number_inpatient", "inpatient_band", "time_in_hospital", "stay_band"]].head()

,number_inpatient,inpatient_band,time_in_hospital,stay_band
0,0.0,0,12.0,10-12 days
1,0.0,0,12.0,10-12 days
2,0.0,0,1.0,1-3 days
3,0.0,0,9.0,7-9 days
4,0.0,0,3.0,1-3 days


## Step 4: Export `patients.csv`

In [4]:
patient_cols = ["encounter_id", "patient_nbr", "race", "gender", "age", "age_mid",
                "admission_type", "discharge_disposition", "admission_source", "medical_specialty",
                "time_in_hospital", "stay_band", "num_lab_procedures", "num_procedures", "num_medications",
                "number_outpatient", "number_emergency", "number_inpatient", "inpatient_band", "total_visits",
                "number_diagnoses", "diag_1_group", "diag_2_group", "diag_3_group",
                "A1Cresult", "insulin", "change", "diabetesMed", "num_active_meds", "num_med_changes",
                "readmitted", "readmitted_30"]

patients = df[patient_cols]
patients.to_csv("../data/powerbi/patients.csv", index=False)

print("patients.csv:", patients.shape)

patients.csv: (69970, 32)


## Step 5: Export `readmission_by_group.csv` (long format)

One table holding the readmission rate for every group of several variables. In Power BI, filter on `variable` to show one chart per variable.

In [5]:
group_vars = ["age", "gender", "race", "diag_1_group", "inpatient_band", "stay_band",
              "A1Cresult", "change", "diabetesMed", "admission_type", "discharge_disposition", "medical_specialty"]

parts = []
for var in group_vars:
    g = df.groupby(var, observed=True)["readmitted_30"].agg(["count", "sum"]).reset_index()
    g.columns = ["group", "patients", "readmitted_30"]
    g["variable"] = var
    g["readmission_rate_pct"] = (g["readmitted_30"] / g["patients"] * 100).round(2)
    parts.append(g)

by_group = pd.concat(parts)[["variable", "group", "patients", "readmitted_30", "readmission_rate_pct"]]
by_group.to_csv("../data/powerbi/readmission_by_group.csv", index=False)

print("readmission_by_group.csv:", by_group.shape)
by_group.loc[by_group["variable"] == "inpatient_band"]

readmission_by_group.csv: (84, 5)


,variable,group,patients,readmitted_30,readmission_rate_pct
0,inpatient_band,0,61779,5015,8.12
1,inpatient_band,1,5794,747,12.89
2,inpatient_band,2,1501,278,18.52
3,inpatient_band,3+,896,237,26.45


## Step 6: Export Model Results (long format)

In [6]:
metrics = pd.read_csv("../data/model_metrics.csv")

# wide (one column per metric) -> long (one row per model x metric)
metrics_long = metrics.melt(id_vars="model", var_name="metric", value_name="value")
metrics_long.to_csv("../data/powerbi/model_metrics.csv", index=False)

top_features = pd.read_csv("../data/top_features.csv")
top_features.to_csv("../data/powerbi/top_features.csv", index=False)

metrics_long

,model,metric,value
0,Logistic Regression,accuracy,0.6581
1,Decision Tree,accuracy,0.6100
2,Logistic Regression,precision,0.1388
3,Decision Tree,precision,0.1277
4,Logistic Regression,recall,0.5402
5,Decision Tree,recall,0.5745
6,Logistic Regression,roc_auc,0.6457
7,Decision Tree,roc_auc,0.6187


## Step 7: Check the Exported Files

In [7]:
for file in sorted(os.listdir("../data/powerbi")):
    t = pd.read_csv("../data/powerbi/" + file)
    print(file, "| rows:", len(t), "| columns:", t.shape[1], "| missing values:", t.isna().sum().sum())

model_metrics.csv | rows: 8 | columns: 3 | missing values: 0


patients.csv | rows: 69970 | columns: 32 | missing values: 0
readmission_by_group.csv | rows: 84 | columns: 5 | missing values: 0
top_features.csv | rows: 20 | columns: 3 | missing values: 0
